In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls "/content/drive/MyDrive"

 archive.zip	   'Google Earth'		    SWaT_Dataset_Attack_v0.csv
 CIKM22-TFAD	    NAB_Archive			    SWaT_Dataset_Normal_v0.csv
 CIKM22-TFAD.zip    nasa_updated.py		    swat_updated.py
'Colab Notebooks'  'Resume Chinmay Gurjar AA.pdf'  'Untitled project.gscript'


In [3]:
!find "/content/drive/MyDrive/NAB_Archive" -maxdepth 2 -type d | sort

/content/drive/MyDrive/NAB_Archive
/content/drive/MyDrive/NAB_Archive/artificialNoAnomaly
/content/drive/MyDrive/NAB_Archive/artificialNoAnomaly/artificialNoAnomaly
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly/artificialWithAnomaly
/content/drive/MyDrive/NAB_Archive/realAdExchange
/content/drive/MyDrive/NAB_Archive/realAdExchange/realAdExchange
/content/drive/MyDrive/NAB_Archive/realAWSCloudwatch
/content/drive/MyDrive/NAB_Archive/realAWSCloudwatch/realAWSCloudwatch
/content/drive/MyDrive/NAB_Archive/realKnownCause
/content/drive/MyDrive/NAB_Archive/realKnownCause/realKnownCause
/content/drive/MyDrive/NAB_Archive/realTraffic
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic
/content/drive/MyDrive/NAB_Archive/realTweets
/content/drive/MyDrive/NAB_Archive/realTweets/realTweets


In [4]:
!find "/content/drive/MyDrive/NAB_Archive" -type f -name "*.csv" | head -30

/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/TravelTime_387.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/occupancy_t4013.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/speed_7578.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/occupancy_6005.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/TravelTime_451.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/speed_6005.csv
/content/drive/MyDrive/NAB_Archive/realTraffic/realTraffic/speed_t4013.csv
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly/artificialWithAnomaly/art_daily_jumpsup.csv
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly/artificialWithAnomaly/art_load_balancer_spikes.csv
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly/artificialWithAnomaly/art_daily_jumpsdown.csv
/content/drive/MyDrive/NAB_Archive/artificialWithAnomaly/artificialWithAnomaly/art_daily_nojump.csv
/content/drive/MyDrive/NAB_Archive/artificialWithAno

In [5]:
nab_root = "/content/drive/MyDrive/NAB_Archive"

In [6]:
!cp "/content/drive/MyDrive/CIKM22-TFAD.zip" /content/
!cp "/content/drive/MyDrive/archive.zip" /content/

!unzip -oq /content/CIKM22-TFAD.zip -d /content/
!unzip -oq /content/archive.zip -d /content/

In [7]:
!pip install torch-optimizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 8.6 MB/s eta 0:00:00


In [8]:
!pip install pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 92.3 MB/s eta 0:00:00


In [9]:
!pip install ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.3 MB/s eta 0:00:00


In [10]:
from pathlib import Path
import re
import os

base = Path("/content/CIKM22-TFAD/tfad")

# 1) general_parser.py
p = base / "examples/article/general_parser.py"
text = p.read_text()
text = text.replace(
    "from pytorch_lightning.utilities.parsing import str_to_bool",
    """def str_to_bool(value):
    if isinstance(value, bool):
        return value
    value = value.lower()
    if value in {"true", "1", "yes", "y", "t"}:
        return True
    if value in {"false", "0", "no", "n", "f"}:
        return False
    raise ValueError(f"Invalid boolean value: {value}")"""
)
p.write_text(text)

# 2) pl_metrics.py
p = base / "src/tfad/utils/pl_metrics.py"
text = p.read_text()
text = text.replace(
    """class CachePredictions(Metric):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)""",
    """class CachePredictions(Metric):
    def __init__(self, *args, **kwargs):
        kwargs.pop("compute_on_step", None)
        super().__init__(*args, **kwargs)"""
)
p.write_text(text)

# 3) tfad.py
p = base / "src/tfad/model/tfad.py"
text = p.read_text()
text = text.replace("cache_preds=CachePredictions(compute_on_step=False),", "cache_preds=CachePredictions(),")
p.write_text(text)

# 4) nasa.py
p = base / "examples/article/nasa.py"
text = p.read_text()

text = text.replace('accelerator="mps"', 'accelerator="cuda" if torch.cuda.is_available() else "cpu"')
text = text.replace("        auto_lr_find=False,\n", "")
text = text.replace("    auto_lr_find=False,\n", "")
text = text.replace(
    "model = TFAD.load_from_checkpoint(ckpt_path)",
    "model = TFAD.load_from_checkpoint(ckpt_path, weights_only=False)"
)
text = text.replace(
    "evaluation_result = trainer.test()",
    "evaluation_result = trainer.test(model=model, datamodule=data_module, ckpt_path=None)"
)
p.write_text(text)

# 5) clear caches
os.system("find /content/CIKM22-TFAD -type d -name '__pycache__' -exec rm -rf {} +")

print("All patches applied.")

All patches applied.


In [11]:
!grep -n "str_to_bool" /content/CIKM22-TFAD/tfad/examples/article/general_parser.py

20:def str_to_bool(value):
99:    parser.add_argument("--normalize_embedding", type=str_to_bool, default=True)
113:    parser.add_argument("--val_labels_adj", type=str_to_bool, default=True)
114:    parser.add_argument("--test_labels_adj", type=str_to_bool, default=True)


In [14]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nasa_fft_final.py \
  --data_dir /content/archive/data/data \
  --model_dir /content/output/smap_model_fft \
  --log_dir /content/output/smap_log_fft \
  --benchmark SMAP \
  --epochs 100

Streaming output truncated to the last 5000 lines.
                                                               train_loss_step: 
Epoch 36/99 ━━╸━━━━━━━━━━━━━━ 6/35 0:00:00 • 0:00:02 19.68it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 36/99 ━━━╺━━━━━━━━━━━━━ 7/35 0:00:00 • 0:00:02 19.66it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 36/99 ━━━╸━━━━━━━━━━━━━ 8/35 0:00:00 • 0:00:02 19.65it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 36/99 ━━━━╺━━━━━━━━━━━━ 9/35 0:00:00 • 0:00:02 19.69it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 36/99 ━━━━╸━━━━━━━━━━━ 10/35 0:00:00 • 0:00:02 19.66it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 36/99 ━━━━━╺━━━━━━━━━━ 11/35 0:00:00 • 0:00:02 19.68

In [15]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nasa_baseline_final.py \
  --data_dir /content/archive/data/data \
  --model_dir /content/output/smap_model_baseline \
  --log_dir /content/output/smap_log_baseline \
  --benchmark SMAP \
  --epochs 100

Streaming output truncated to the last 5000 lines.
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━╺━━━━━ 22/35 0:00:01 • 0:00:01 19.89it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━╸━━━━━ 23/35 0:00:01 • 0:00:01 19.89it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━╸━━━━━ 24/35 0:00:01 • 0:00:01 19.89it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━━╺━━━━ 25/35 0:00:01 • 0:00:01 19.90it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━━╸━━━━ 26/35 0:00:01 • 0:00:01 19.91it/s v_num: 0.000     
                                                               train_loss_step: 
Epoch 39/99 ━━━━━━━━━━━━╺━━━ 27/35 0:00:01 • 0:00:01 19.91

In [16]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nasa_fft_final.py \
  --data_dir /content/archive/data/data \
  --model_dir /content/output/msl_model \
  --log_dir /content/output/msl_log \
  --benchmark MSL \
  --epochs 100

Seed set to 123
100% 27/27 [00:00<00:00, 1631.65it/s]
Total channels after adding local FFT features: 5
Validation series count: 13
Test series count: 14
Validation set series 0 positive labels: 180
Validation set series 1 positive labels: 1140
Validation set series 2 positive labels: 1140
Validation set series 3 positive labels: 10
Validation set series 4 positive labels: 130
Validation set series 5 positive labels: 68
Validation set series 6 positive labels: 25
Validation set series 7 positive labels: 420
Validation set series 8 positive labels: 250
Validation set series 9 positive labels: 250
Validation set series 10 positive labels: 300
Validation set series 11 positive labels: 20
Validation set series 12 positive labels: 310
Test set series 0 positive labels: 135
Test set series 1 positive labels: 120
Test set series 2 positive labels: 250
Test set series 3 positive labels: 70
Test set series 4 positive labels: 150
Test set series 5 positive labels: 220
Test set series 6 positive 

In [17]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nasa_baseline_final.py \
  --data_dir /content/archive/data/data \
  --model_dir /content/output/msl_model_baseline \
  --log_dir /content/output/msl_log_baseline \
  --benchmark MSL \
  --epochs 100

Seed set to 123
100% 27/27 [00:00<00:00, 1548.79it/s]
Validation series count: 13
Test series count: 14
Validation set series 0 positive labels: 180
Validation set series 1 positive labels: 1140
Validation set series 2 positive labels: 1140
Validation set series 3 positive labels: 10
Validation set series 4 positive labels: 130
Validation set series 5 positive labels: 68
Validation set series 6 positive labels: 25
Validation set series 7 positive labels: 420
Validation set series 8 positive labels: 250
Validation set series 9 positive labels: 250
Validation set series 10 positive labels: 300
Validation set series 11 positive labels: 20
Validation set series 12 positive labels: 310
Test set series 0 positive labels: 135
Test set series 1 positive labels: 120
Test set series 2 positive labels: 250
Test set series 3 positive labels: 70
Test set series 4 positive labels: 150
Test set series 5 positive labels: 220
Test set series 6 positive labels: 110
Test set series 7 positive labels: 180

In [18]:
!find "/content/drive/MyDrive/NAB_Archive" -type f | grep -i "json"

In [19]:
!git clone https://github.com/numenta/NAB.git /content/NAB

Cloning into '/content/NAB'...
remote: Enumerating objects: 7119, done.
remote: Counting objects: 100% (731/731), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 7119 (delta 564), reused 502 (delta 502), pack-reused 6388 (from 1)
Receiving objects: 100% (7119/7119), 86.16 MiB | 32.15 MiB/s, done.
Resolving deltas: 100% (4983/4983), done.


In [20]:
!ls /content/NAB/labels

combined_labels.json  combined_windows.json  combined_windows_tiny.json  raw


In [21]:
json_path = "/content/NAB/labels/combined_windows.json"

In [22]:
import json
import numpy as np
import pandas as pd

file_path = "/content/drive/MyDrive/NAB_Archive/realAWSCloudwatch/realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv"
json_path = "/content/NAB/labels/combined_windows.json"

df = pd.read_csv(file_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)
df["value"] = df["value"].ffill().bfill()

with open(json_path, "r") as f:
    windows_dict = json.load(f)

file_key = "realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv"
windows = windows_dict[file_key]

labels = np.zeros(len(df), dtype=np.float32)

for start, end in windows:
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
    labels[mask.to_numpy()] = 1.0

values = df["value"].to_numpy(dtype=np.float32)

print(values.shape, labels.shape)
print("Anomalies:", labels.sum())

(4032,) (4032,)
Anomalies: 402.0


In [31]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nab_fft.py \
  --data_dir /content/nab_as_swat \
  --model_dir /content/output/nab_model_fft_local \
  --log_dir /content/output/nab_log_fft_local \
  --epochs 100 \
  --window_length 150 \
  --suspect_window_length 5 \
  --fft_window 64

Seed set to 123

Loading SWaT dataset...

Loaded normal shape: (2016, 3)
Loaded attack shape: (2016, 3)

...SWaT dataset loaded succesfully.

Total channels after adding FFT features: 5
Validation series count: 0
Test series count: 1
Test set series 0 positive labels: 402
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'distance' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['distance'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'classification_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['classification_loss'])`.
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametriz

In [32]:
!PYTHONPATH=/content/CIKM22-TFAD/tfad/src python /content/CIKM22-TFAD/tfad/examples/article/nab_baseline.py \
  --data_dir /content/nab_as_swat \
  --model_dir /content/output/nab_model_baseline \
  --log_dir /content/output/nab_log_baseline \
  --epochs 100 \
  --window_length 150 \
  --suspect_window_length 5

Seed set to 123

Loading SWaT dataset...

Loaded normal shape: (2016, 3)
Loaded attack shape: (2016, 3)

...SWaT dataset loaded succesfully.

Validation series count: 0
Test series count: 1
Test set series 0 positive labels: 402
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'distance' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['distance'])`.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/parsing.py:213: Attribute 'classification_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['classification_loss'])`.
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(modu